# Check coverage and completeness of all raw gauge data

Information extracted for each gauge location and stored in the `location_analytics` dictionary includes:
* Date of first data
* Date of last data
* Monitoring duration (first to last data)
* All intentional timesteps for each parameter monitored, including the number of measurements made with each timestep (timesteps vary between gauges, and in some cases different timesteps were used at different times at a single gauge)
* The maximum intentional timestep for each parameter monitored
* Any data gaps identified (including start, end and duration for each)
* Raw data completeness for each parameter (with respect to the monitoring period of the gauge)

In [ ]:
# Imports
import os
import pandas as pd
import numpy as np
import math

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle

import blue_heart_data_handling as bhdh

In [ ]:
# General settings

# Whether to save plots
save_flag = True

# Gauge index
path_index_telemetry_data = '../index_telemetry_data.csv'
df_index = pd.read_csv(path_index_telemetry_data)

# Folder to save plots in
path_output_folder = 'outputs/3_raw_data_completeness'
if not os.path.isdir(path_output_folder):
    os.makedirs(path_output_folder)

# Plot settings
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 10,
    'axes.titlesize': 10,
    'axes.labelsize': 10,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10
})

## Calculate coverage and data completeness for all gauges and timeseries

In [ ]:
# Function to identify data gaps.
# Adapted from https://towardsdatascience.com/handling-gaps-in-time-series-dc47ae883990/.

def get_missing_data_periods(
        df: pd.DataFrame,               # Datetime indexed dataframe.
        timestep_expected_mins: int,    # Max expected data timestep. Timesteps more than 10% greater will be flagged as gaps.                          
        column_value: str = None,
        index_name='timestamp'):
    df = df.copy()

    # Analyse first column by default
    if not column_value:
        column_value = df.columns[0]

    # Round datetime index to nearest minute
    df.index = df.index.round('1min')

    # Drop duplicate times (keep last of each)
    df = df[~df.index.duplicated(keep='last')]

    # Resample to 1 minute frequency
    df = df.asfreq('1min')

    # Reset index
    df.reset_index(inplace=True)

    # Indexer for nonmissing value on data
    is_not_nan = ~df[column_value].isna()

    # Auxiliary indexer to group sequences
    # diff and cumsum aggregate nonmissing sequences
    group_idx = is_not_nan.diff().cumsum().astype(float).fillna(0)

    # Not-missing counter
    not_nan_counts = is_not_nan.groupby(group_idx).sum()

    # Instantiate sequence lengths DataFrame and retrieve position indices and time
    sequences_df = pd.Series(
        np.arange(len(df))).groupby(group_idx).agg(['min', 'max'])
    sequences_df['seq_start_time'] = sequences_df['min'].map(df[index_name])
    sequences_df['seq_end_time'] = sequences_df['max'].map(df[index_name])
    sequences_df['duration'] = sequences_df.apply(
        lambda x: x['seq_end_time'] - x['seq_start_time'], axis=1)
    sequences_df['not_nan_count'] = not_nan_counts
    sequences_df['nan_count'] = (
        sequences_df['max'] - sequences_df['min']) - (sequences_df['not_nan_count'] - 1)

    # Assert sum of sequence lengths == total series length
    assert sum(sequences_df[['not_nan_count', 'nan_count']].sum()) == df.shape[0]

    # Tidy up
    sequences_df.rename(columns={'min': 'seq_start_idx', 'max': 'seq_end_idx'}, inplace=True)
    sequences_df.sort_values('seq_start_idx', inplace=True)
    sequences_df.reset_index(drop=True, inplace=True)

    # Threshold for gap to be considered missing data (1.1 x expected frequency)
    threshold_missing = int(timestep_expected_mins * 1.1)

    # Add continuous Missing Values (CMV) flag
    sequences_df['data_missing'] = (sequences_df['nan_count'] > threshold_missing).astype(int)

    # Return data gaps only
    return sequences_df[sequences_df['data_missing'] > 0].drop(
        columns=['seq_start_idx', 'seq_end_idx', 'not_nan_count', 'nan_count', 'data_missing']
    )

In [ ]:
# Load all raw data

dfs_raw = bhdh.get_gauge_data(
    data_type_option=bhdh.DataTypeOptions.Raw,
)

In [ ]:
def gauge_type_from_name(gauge_name: str, df_index: pd.DataFrame):
    return df_index[df_index['locationName'] == gauge_name]['gaugeType'].unique()[0]

In [ ]:
# Analyse all gauges

location_analytics = {}

for location in dfs_raw.keys():
    print(f'Analysing data from {location}')
    location_analytics[location] = {}
    
    # Store gauge type
    location_analytics[location]['gauge_type'] = gauge_type_from_name(location, df_index)

    # Analyse each parameter at gauge
    parameters = dfs_raw[location].keys()
    location_analytics[location]['date_start'] = {}
    location_analytics[location]['date_end'] = {}
    location_analytics[location]['duration_monitoring'] = {}
    location_analytics[location]['timesteps_intentional'] = {}
    location_analytics[location]['timesteps_max'] = {}
    location_analytics[location]['timesteps_mode'] = {}
    location_analytics[location]['data_gaps'] = {}
    location_analytics[location]['data_gaps_duration'] = {}
    location_analytics[location]['completeness_monitoring_period_percent'] = {}

    for parameter in parameters:
        df = dfs_raw[location][parameter].dropna()
        parameter_name = parameter.value

        # Data coverage
        date_start = df.index.min()
        date_end = df.index.max()
        duration_monitoring = date_end - date_start
        location_analytics[location]['date_start'][parameter_name] = date_start
        location_analytics[location]['date_end'][parameter_name] = date_end
        location_analytics[location]['duration_monitoring'][parameter_name] = duration_monitoring

        # All intentional timesteps
        timesteps_intentional = bhdh.get_intentional_timesteps(df)
        location_analytics[location]['timesteps_intentional'][parameter_name] = timesteps_intentional

        # Maximum intentional timestep
        timesteps_intentional_max = bhdh.get_max_intentional_timestep_minutes(timesteps_intentional)
        location_analytics[location]['timesteps_max'][parameter_name] = timesteps_intentional_max

        # Most common intentional timestep
        timesteps_intentional_mode = bhdh.get_modal_intentional_timestep_minutes(timesteps_intentional)
        location_analytics[location]['timesteps_mode'][parameter_name] = timesteps_intentional_mode

        # Identify data gaps
        df_gaps = get_missing_data_periods(
            df=df,
            timestep_expected_mins=timesteps_intentional_max
        )
        location_analytics[location]['data_gaps'][parameter_name] = df_gaps

        # Calculate total duration of missing data during the monitoring period for the gauge
        data_gaps_duration = df_gaps['duration'].sum()
        location_analytics[location]['data_gaps_duration'][parameter_name] = data_gaps_duration

        # Calculate data completeness (with respect to monitoring period for the gauge) (%)
        dataset_completeness_percent = 100 * (1 - data_gaps_duration / duration_monitoring)
        location_analytics[location]['completeness_monitoring_period_percent'][parameter_name] = dataset_completeness_percent



In [ ]:
# Convert to dataframe
records = [{'Location': location, **values} for location, values in location_analytics.items()]
df_location_analytics = pd.json_normalize(records, sep='_')

## Visualise coverage and data completeness for all gauges and timeseries

In [ ]:
# Identify all parameters monitored
parameters_all = [list(location_analytics[location]['timesteps_intentional'].keys()) for location in location_analytics]
parameters_unique = list(set(item for sublist in parameters_all for item in sublist))
parameters_unique

In [ ]:
# Function to split dataframe into chunks to limit plot sizes
def split_dataframe(df, max_rows):
    total_rows = len(df)
    
    if total_rows <= max_rows:
        return [df]
    
    # Number of chunks needed
    num_chunks = math.ceil(total_rows / max_rows)
    
    # Target size for each chunk (as equal as possible)
    base_size = total_rows // num_chunks
    remainder = total_rows % num_chunks  # first 'remainder' chunks get +1 row
    
    chunks = []
    start = 0
    
    for i in range(num_chunks):
        size = base_size + (1 if i < remainder else 0)
        end = start + size
        chunks.append(df.iloc[start:end])
        start = end
    
    return chunks 

In [ ]:
# Plot data coverage for each timeseries (each gauge type and parameter on separate plot)

threshold_gap_duration = pd.Timedelta(hours=6)  # Only plot data gaps larger than this

start_min = min([df_location_analytics[f'date_start_{parameter}'].min() for parameter in parameters_unique])
end_max = max([df_location_analytics[f'date_start_{parameter}'].max() for parameter in parameters_unique])

# Identify all gauge types
for gauge_type in df_location_analytics['gauge_type'].unique():
    df_gauge_type = df_location_analytics[df_location_analytics['gauge_type'] == gauge_type]

    for parameter in parameters_unique:
        print(f'Plotting data coverage for gauge type = {gauge_type}, parameter = {parameter} timeseries')
    
        # Filter to locations at which this parameter is monitored
        data_gaps_column = f'data_gaps_{parameter}'
        df = df_gauge_type.copy().dropna(subset=[data_gaps_column])

        if df.shape[0] > 0:
    
            # Limit the maximum number of locations shown in each plot
            chunks_df = split_dataframe(df, max_rows=30)
            num_chunks = len(chunks_df)
        
            for chunk_number, chunk_df in enumerate(chunks_df):
        
                num_locations = chunk_df.shape[0]
        
                # Create subplot for each location
                fig, axes = plt.subplots(
                    nrows=num_locations,
                    ncols=1,
                    figsize=(10, 0.15 * num_locations + 0.5),
                    sharex=True
                )
        
                # Ensure axes is iterable when only one subplot exists
                if num_locations == 1:
                    axes = [axes]
        
                for ax, (_, location_row) in zip(axes, chunk_df.iterrows()):
        
                    # Configure axes
                    ax.set_xlim(start_min, end_max)
                    ax.set_ylim(0, 1)
                    ax.set_yticks([])
        
                    # Remove unnecessary spines
                    ax.spines['top'].set_visible(False)
                    ax.spines['right'].set_visible(False)
                    ax.spines['left'].set_visible(False)
        
                    # Fill entire plot background green
                    ax.add_patch(
                        Rectangle(
                            (start_min, 0),
                            width=end_max - start_min,
                            height=1,
                            facecolor='green',
                            edgecolor='none'
                        )
                    )
        
                    # Identify periods before start and after end of data collection
                    grey_periods = [
                        (
                            start_min,
                            max(start_min, location_row[f'date_start_{parameter}'])
                        ),
                        (
                            min(end_max, location_row[f'date_end_{parameter}']),
                            end_max
                        )
                    ]
        
                    for x0, x1 in grey_periods:
                        if x1 > x0:
                            ax.add_patch(
                                Rectangle(
                                    (x0, 0),
                                    width=x1 - x0,
                                    height=1,
                                    facecolor='grey',
                                    edgecolor='none'
                                )
                            )
        
                    # Identify gaps during data collection
                    df_data_gaps = location_row[data_gaps_column].copy()
                    df_data_gaps = df_data_gaps[
                        df_data_gaps['duration'] > threshold_gap_duration
                    ]
        
                    for _, gap_row in df_data_gaps.iterrows():
                        x0 = gap_row['seq_start_time']
                        x1 = gap_row['seq_end_time']
        
                        ax.add_patch(
                            Rectangle(
                                (x0, 0),
                                width=x1 - x0,
                                height=1,
                                facecolor='grey',
                                edgecolor='none'
                            )
                        )
        
                    # Add location label on left side
                    ax.text(
                        -0.02,                     # slightly left of axis
                        0.5,
                        location_row['Location'],
                        ha='right',
                        va='center',
                        fontsize=8,
                        transform=ax.transAxes     # use axis coordinates
                    )
        
                # Format x-axis
                axes[-1].set_xlabel('Date')
                axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
        
                # Rotate dates for readability
                plt.xticks(rotation=45)
        
                # Adjust layout and margins
                plt.subplots_adjust(
                    left=0.3,
                    right=0.95,
                    top=0.98,
                    bottom=0.12,
                    hspace=0
                )
        
                # Save figure
                if save_flag:
                    output_path = (
                        f'{path_output_folder}/Data coverage {gauge_type}-{parameter} '
                        f'(part {chunk_number + 1} of {num_chunks}).png'
                    )
                    fig.savefig(output_path, dpi=600, bbox_inches='tight')
                else:
                    plt.show()
                
                plt.close(fig)

## Summarise duration of data available across all gauges

In [ ]:
# For each parameter at each location, calculate duration of complete data available
for parameter in parameters_unique:
    df_location_analytics[f'duration_complete_{parameter}'] = df_location_analytics[f'duration_monitoring_{parameter}'] - df_location_analytics[f'data_gaps_duration_{parameter}']

# Gauge level monitoring duration (any parameter)
df_location_analytics['duration_monitoring'] = df_location_analytics.filter(like='date_end_').max(axis=1) - df_location_analytics.filter(like='date_start_').min(axis=1)
# Gauge level duration of complete data available (based on minimum across all timeseries recorded at gauge)
df_location_analytics['duration_complete'] = df_location_analytics.filter(like='duration_complete_').min(axis=1)
# Gauge level data completeness (based on minimum across all timeseries recorded at gauge)
df_location_analytics['completeness_monitoring_period_percent'] = df_location_analytics.filter(like='completeness_monitoring_period_percent_').min(axis=1)

df_location_analytics[['Location', 'duration_monitoring', 'duration_complete', 'completeness_monitoring_period_percent']]

In [ ]:
# Plot distribution of monitoring durations and complete data durations across all locations / gauges

df = df_location_analytics.copy()

# Convert Timedelta columns to numeric values (e.g. days)
monitoring_days = df['duration_monitoring'].dt.total_seconds() / 86400

completeness = df['completeness_monitoring_period_percent']

# Create horizontal boxplots
fig, (ax1, ax2) = plt.subplots(
    2, 1,
    figsize=(10, 2.5),
    sharex=False,
    # gridspec_kw={'height_ratios': [1, 1]}
)

# Top subplot: durations
ax1_data = monitoring_days.dropna()
ax1.violinplot(
    ax1_data,
    vert=False,
    showextrema=False,
    widths=0.8
)
ax1.boxplot(
    ax1_data,
    vert=False,
    widths=0.15,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor='none', edgecolor='black'),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black')
)
ax1.set_xlabel('Duration (days)')
ax1.set_yticklabels(['Monitoring period'])
ax1.set_title('a)')

# Bottom subplot: completeness
ax2_data = completeness.dropna()
ax2.violinplot(
    ax2_data,
    vert=False,
    showextrema=False,
    widths=0.8,
)
ax2.boxplot(
    ax2_data,
    vert=False,
    widths=0.15,
    patch_artist=True,
    showfliers=False,
    boxprops=dict(facecolor='none', edgecolor='black'),
    medianprops=dict(color='red', linewidth=2),
    whiskerprops=dict(color='black'),
    capprops=dict(color='black')
)
ax2.set_xlabel('Completeness (%)')
ax2.set_yticks([1])
ax2.set_yticklabels(['Dataset\ncompleteness'])
ax2.set_title('b)')



ax1.set_xlim(0, 1.05 * ax1_data.max())
ax2.set_xlim(0, 100)

plt.tight_layout()
plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Data durations summary.png', dpi=600, bbox_inches='tight')

plt.close(fig)


## Summarise number of gauges with data available each day

In [ ]:
# Calculate number of gauges with data (from any timeseries) available each day. Split by gauge type

# Gauge-level data start and end dates
df_location_analytics['gauge_start'] = df_location_analytics.filter(like='date_start').min(axis=1)
df_location_analytics['gauge_end'] = df_location_analytics.filter(like='date_end').max(axis=1)

# Catchment monitoring start and end dates
all_dates = pd.date_range(
    df_location_analytics['gauge_start'].min(),
    df_location_analytics['gauge_end'].max(),
    freq='D'
)

gap_cols = [
    c for c in df_location_analytics.columns
    if c.startswith('data_gaps_') and not c.startswith('data_gaps_duration_')
]


def build_availability(dates, start, end, gaps_df):
    """
    dates: DatetimeIndex
    start/end: coverage window
    gaps_df: dataframe with seq_start_time, seq_end_time
    """
    mask = (dates >= start) & (dates <= end)
    if gaps_df is None:
        return mask
    if not isinstance(gaps_df, pd.DataFrame):
        return mask  # catches NaN, float, etc.
    if gaps_df.empty:
        return mask
    for _, g in gaps_df.iterrows():
        mask &= ~((dates >= g['seq_start_time']) & (dates <= g['seq_end_time']))

    return mask


location_availability = {}
for _, row in df_location_analytics.iterrows():
    location = row['Location']
    date_start, date_end = row['gauge_start'], row['gauge_end']

    # start with "no data available"
    location_mask = pd.Series(False, index=all_dates)

    for col in gap_cols:
        gaps_df = row[col]

        # Assume availability for this data type within coverage window
        available = (all_dates >= date_start) & (all_dates <= date_end)

        # Subtract gaps
        available = build_availability(all_dates, date_start, date_end, gaps_df)

        # Union across parameters
        location_mask |= available

    location_availability[location] = location_mask

# Aggregate number of locations / gauges with any data available each day (separated by gauge type)
df_availability = pd.DataFrame(location_availability, index=all_dates)
daily_counts_gauge_type_list = []
for gauge_type in df_location_analytics['gauge_type'].unique():
    locations = df_location_analytics[df_location_analytics['gauge_type'] == gauge_type]['Location'].to_list()
    df_availability_gauge_type = df_availability[locations]
    daily_counts_gauge_type = df_availability_gauge_type.sum(axis=1).rename(gauge_type)
    daily_counts_gauge_type_list.append(daily_counts_gauge_type)
df_daily_counts = pd.concat(daily_counts_gauge_type_list, axis=1)
df_daily_counts

In [ ]:
# Plot results
fig, ax = plt.subplots(figsize=(10, 3))

# Colour palette
colors = [
    "#0072B2",  # blue
    "#E69F00",  # orange
    "#009E73",  # green
    "#CC79A7",  # purple
    "#D55E00",  # vermillion
]

# Plot stacked bars
bottom = np.zeros(len(df_daily_counts))
for i, gauge_type in enumerate(df_daily_counts.columns):
    ax.bar(
        df_daily_counts.index,
        df_daily_counts[gauge_type],
        width=1,
        edgecolor='none',
        bottom=bottom,
        label=gauge_type,
        color=colors[i % len(colors)]
    )
    # Update bottom for next stack
    bottom = bottom + df_daily_counts[gauge_type].values

ax.set_xlabel('Date')
ax.set_ylabel('Number of gauges\nwith data')
plt.grid(linestyle = '--', linewidth = 0.5)
fig.autofmt_xdate()
ax.set_xlim([df_daily_counts.index.min(), df_daily_counts.index.max()])
ax.set_ylim([0, df_daily_counts.sum(axis=1).max()*1.05])
ax.legend(
    loc="upper left",
    frameon=True,
    fontsize=9
)
plt.tight_layout()

plt.show()

if save_flag:
    fig.savefig(f'{path_output_folder}/Number of gauges with data.png', dpi=600, bbox_inches='tight')

plt.close(fig)

In [ ]:
# Summary stats: Longest continuous period with data available from at least X gauges? (duration and dates)

for count_threshold in [300, 400]:
    # Total gauges of all types with data each day
    daily_total_gauges = df_daily_counts.sum(axis=1)
    # Days with gauge count above threshold
    mask = daily_total_gauges > count_threshold
    # Groups of consecutive days with count above threshold
    groups = (mask != mask.shift()).cumsum()[mask]
    # Calculate group lengths
    runs = (
        mask[mask]
        .groupby(groups[mask])
        .agg(
            start=lambda x: x.index.min(),
            end=lambda x: x.index.max(),
            length=lambda x: x.index.max() - x.index.min()
            )
    )
    longest = runs.loc[runs['length'].idxmax()]
    start = longest['start']
    end = longest['end']
    duration_days = longest['length'].days
    duration_months = duration_days / (365 / 12)
    print(f'Longest period with date from at least {count_threshold} gauges: {start} - {end} ({duration_days} days / {round(duration_months, 1)} months')

In [ ]:
# Summary stats: Min, median and max monitoring durations for different gauges
print(f'Minimum gauge level monitoring period: {df_location_analytics['duration_monitoring'].min().days} days')
print(f'Median gauge level monitoring period: {df_location_analytics['duration_monitoring'].median().days} days')
print(f'Maximum gauge level monitoring period: {df_location_analytics['duration_monitoring'].max().days} days')

In [ ]:
# Summary stats: Min, median and max down times for different gauges

df_location_analytics['downtime_percent'] = 100 - df_location_analytics['completeness_monitoring_period_percent']

print(f'Minimum gauge level downtime: {round(df_location_analytics['downtime_percent'].min(), 1)} %')
print(f'Median gauge level downtime: {round(df_location_analytics['downtime_percent'].median(), 1)} %')
print(f'Maximum gauge level downtime: {round(df_location_analytics['downtime_percent'].max(), 1)} %')